# Nạp kết quả TN1 từ tệp nén trên Drive

Notebook **chỉ đọc Drive**. Không train, không clone mã, không sửa gì trên Drive.

Ba cấu hình baseline của TN1 đã train xong từ trước, nhưng tệp kết quả chỉ được
nén lên Drive chứ chưa bao giờ vào git. Notebook này lấy chúng về, xếp lại đúng
layout `runs/` của nhánh nộp, rồi cho tải xuống một tệp `.tar.gz`.

| cấu hình | tệp nén cần | ghi chú |
|---|---|---|
| LSTM-352 | `tn1_lstm_mse_corr0.9_seed2.zip` | tệp nén tích luỹ — bản seed cuối chứa cả ba seed |
| LSTM-67 | `tn1_lstm_h67_mse_corr0.9_seed2.zip` | như trên |
| CNN-LSTM-58 | `tn1_cnn_lstm_h58.zip` | |

`DS-TCN-C64-RF61` đã nằm sẵn trong git, không cần lấy lại.

Chạy lần lượt từ trên xuống.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Bốn cấu hình và tên thư mục đích

`config_id` dưới đây là tên thật `run_cv.py` đã sinh ra lúc train. Đối chiếu
đúng chuỗi này, không đoán theo tên tệp nén.

In [ ]:
SRC = "/content/drive/MyDrive/mobivital"

# config_id (bỏ phần _seed<N>)  ->  tên thư mục trong runs/tn1/
CAU_HINH = {
    "lstm_mse_corr0.9":                              "LSTM-352",
    "lstm_h67_mse_corr0.9":                          "LSTM-67",
    "cnn_lstm_h58_c32_k5_mse_corr0.9":               "CNN-LSTM-58",
    "ds_tcn_c64_k3_n4_none_do0.2_dpel_mse_corr0.9":  "DS-TCN-C64-RF61",
}

# Điểm phải ra sau khi nạp. Lấy từ bảng kết quả của đồ án; nếu tính lại từ
# scores.csv mà lệch quá NGUONG thì tệp nén sai hoặc thiếu fold.
MONG_DOI = {
    "LSTM-352":        (0.756998, 0.004141),
    "LSTM-67":         (0.753208, 0.001966),
    "CNN-LSTM-58":     (0.752666, 0.003749),
    "DS-TCN-C64-RF61": (0.760878, 0.003095),
}
NGUONG = 1e-4

FOLDS = ["val_AB", "val_CE", "val_DF", "val_KL"]

## 3. Liệt kê tệp nén đang có trên Drive

In [ ]:
import glob, os

CAN = ["tn1_lstm_mse_corr0.9_seed2.zip",
       "tn1_lstm_h67_mse_corr0.9_seed2.zip",
       "tn1_cnn_lstm_h58.zip"]

# Bản dự phòng nếu tệp chính không còn: tệp nén tích luỹ nên seed nhỏ hơn
# chứa ít seed hơn, nhưng vẫn dùng được nếu gộp nhiều tệp.
DU_PHONG = ["tn1_lstm.zip",
            "tn1_lstm_mse_corr0.9_seed1.zip",
            "tn1_lstm_h67_mse_corr0.9_seed0.zip",
            "tn1_lstm_h67_mse_corr0.9_seed1.zip",
            "tn1_cnn_lstm_h58_c32_k5_mse_corr0.9_seed0.zip",
            "tn1_cnn_lstm_h58_c32_k5_mse_corr0.9_seed1.zip",
            "tn1_cnn_lstm_h58_c32_k5_mse_corr0.9_seed2.zip"]

def mb(p):
    return os.path.getsize(p) / 1e6

print("CẦN")
co = []
for ten in CAN:
    p = os.path.join(SRC, ten)
    if os.path.exists(p):
        co.append(p); print("   %8.1f MB  %s" % (mb(p), ten))
    else:
        print("   %8s     %s   KHÔNG THẤY" % ("-", ten))

print()
print("DỰ PHÒNG có sẵn")
for ten in DU_PHONG:
    p = os.path.join(SRC, ten)
    if os.path.exists(p):
        co.append(p); print("   %8.1f MB  %s" % (mb(p), ten))

print()
print("sẽ giải", len(co), "tệp nén")

## 4. Giải nén và xếp lại layout

Tệp nén của `save_results.py` có dạng phẳng:

```
tn1/<config_id>_<fold>/final.pth
tn1/<config_id>_<fold>/curve.csv
tn1/scores_<config_id>_<fold>.csv
```

Nhánh nộp dùng dạng lồng theo cấu hình:

```
runs/tn1/<tên>/seed<N>/<fold>/{final.pth, curve.csv, scores.csv}
```

Ô này làm đúng phép đổi đó. Tệp nào đã có rồi thì **so byte**; khác byte là báo
xung đột chứ không đè lặng lẽ.

In [ ]:
import hashlib, re, shutil, subprocess, tempfile

DICH = "/content/runs_TN1/tn1"
os.makedirs(DICH, exist_ok=True)

def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

xung_dot, da_chep = [], 0

def chep(nguon, dich):
    global da_chep
    if os.path.exists(dich):
        if sha(nguon) != sha(dich):
            xung_dot.append(dich)
        return
    os.makedirs(os.path.dirname(dich), exist_ok=True)
    shutil.copy2(nguon, dich)
    da_chep += 1

def tach(ten):
    """'lstm_h67_mse_corr0.9_seed1_val_AB' -> ('LSTM-67', 1, 'val_AB') hoặc None."""
    m = re.match(r"^(.*)_seed(\d+)_(val_[A-Z]{2})$", ten)
    if not m:
        return None
    goc, seed, fold = m.group(1), int(m.group(2)), m.group(3)
    if goc not in CAU_HINH:
        return None
    return CAU_HINH[goc], seed, fold

tam = tempfile.mkdtemp()
for i, z in enumerate(co):
    subprocess.run(["unzip", "-oq", z, "-d", os.path.join(tam, str(i))], check=True)

bo_qua = set()
for thu_muc, _, ten_tep in os.walk(tam):
    nhan = os.path.basename(thu_muc)

    # thư mục <config_id>_<fold>/ chứa final.pth và curve.csv
    r = tach(nhan)
    if r:
        cau_hinh, seed, fold = r
        for t in ten_tep:
            if t in ("final.pth", "curve.csv"):
                chep(os.path.join(thu_muc, t),
                     "%s/%s/seed%d/%s/%s" % (DICH, cau_hinh, seed, fold, t))
        continue

    # tệp scores_<config_id>_<fold>.csv nằm ngang hàng
    for t in ten_tep:
        if t.startswith("scores_") and t.endswith(".csv"):
            r = tach(t[len("scores_"):-len(".csv")])
            if r:
                cau_hinh, seed, fold = r
                chep(os.path.join(thu_muc, t),
                     "%s/%s/seed%d/%s/scores.csv" % (DICH, cau_hinh, seed, fold))
            else:
                bo_qua.add(t)

print("chép", da_chep, "tệp")
if bo_qua:
    print("bỏ qua", len(bo_qua), "tệp scores của cấu hình ngoài phạm vi nhánh này")
if xung_dot:
    print()
    print("XUNG ĐỘT — cùng đường dẫn, khác nội dung:")
    for d in xung_dot:
        print("   ", d)
else:
    print("không xung đột")

## 5. Đủ chưa — mỗi cấu hình phải có 3 seed × 4 fold × 3 tệp

In [ ]:
thieu = []
print("%-18s %s" % ("cấu hình", "seed0        seed1        seed2"))
print("-" * 60)
for ten in ["LSTM-352", "LSTM-67", "CNN-LSTM-58"]:
    dong = []
    for seed in (0, 1, 2):
        n = 0
        for fold in FOLDS:
            d = "%s/%s/seed%d/%s" % (DICH, ten, seed, fold)
            for t in ("final.pth", "curve.csv", "scores.csv"):
                if os.path.exists(os.path.join(d, t)):
                    n += 1
                else:
                    thieu.append("%s/seed%d/%s/%s" % (ten, seed, fold, t))
        dong.append("%2d/12 tệp" % n)
    print("%-18s %s" % (ten, "   ".join(dong)))

print()
if thieu:
    print("THIẾU", len(thieu), "tệp:")
    for t in thieu[:20]:
        print("   ", t)
else:
    print("ĐỦ — 3 seed × 4 fold × 3 tệp cho cả ba cấu hình")

## 6. Kiểm điểm — tính lại từ `scores.csv`, so với bảng của đồ án

Đây là phép kiểm quan trọng nhất. Điểm chính thức là **Pearson macro theo
người**: trung bình theo từng người trước, rồi mới trung bình các người. Tính
lại từ điểm từng buổi ghi, không đọc lại số đã ghi sẵn ở đâu.

In [ ]:
import csv, statistics

def macro_cua_fold(duong_dan):
    rows = list(csv.DictReader(open(duong_dan)))
    theo_nguoi = {}
    for r in rows:
        theo_nguoi.setdefault(r["user"], []).append(float(r["pearson"]))
    return statistics.mean(statistics.mean(v) for v in theo_nguoi.values()), len(rows)

print("%-18s %10s %10s %10s   %10s %10s   %s"
      % ("cấu hình", "seed0", "seed1", "seed2", "cv_mean", "seed_std", "so bảng"))
print("-" * 96)

tat_ca_dat = True
for ten in ["LSTM-352", "LSTM-67", "CNN-LSTM-58", "DS-TCN-C64-RF61"]:
    goc = DICH + "/" + ten
    if not os.path.isdir(goc):
        if ten == "DS-TCN-C64-RF61":
            print("%-18s  bỏ qua — đã nằm sẵn trong git, không lấy lại" % ten)
        else:
            print("%-18s  KHÔNG CÓ DỮ LIỆU — thiếu tệp nén của cấu hình này" % ten)
            tat_ca_dat = False
        continue
    per_seed = []
    for seed in (0, 1, 2):
        diem = [macro_cua_fold("%s/seed%d/%s/scores.csv" % (goc, seed, f))[0]
                for f in FOLDS]
        per_seed.append(statistics.mean(diem))
    tb, sd = statistics.mean(per_seed), statistics.stdev(per_seed)
    mong_tb, mong_sd = MONG_DOI[ten]
    dat = abs(tb - mong_tb) < NGUONG and abs(sd - mong_sd) < NGUONG
    tat_ca_dat &= dat
    print("%-18s %10.6f %10.6f %10.6f   %10.6f %10.6f   %s"
          % (ten, per_seed[0], per_seed[1], per_seed[2], tb, sd,
             "ĐẠT" if dat else "LỆCH (bảng %.6f +- %.6f)" % (mong_tb, mong_sd)))

print()
print("TẤT CẢ KHỚP BẢNG" if tat_ca_dat else "CÓ DÒNG LỆCH — xem lại tệp nén")

## 7. Gộp `summary.csv`

Lấy các dòng của bốn cấu hình từ `summary.csv` bên trong tệp nén, bỏ cột đã gỡ
khỏi nhánh nộp (`revin`, `test_ghij_macro`) và xoá trắng cột `device` — tên đời
GPU không đổi kết quả nào mà lại dính vào mọi bảng.

In [ ]:
COT = ["run_id", "timestamp", "git_commit", "device",
       "experiment", "model", "loss", "alpha",
       "corr_threshold", "seed", "fold", "val_users",
       "n_params", "n_train_windows", "epochs",
       "train_mse", "train_pearson", "train_loss", "minutes_train", "resumed",
       "score_macro", "score_micro", "score_std", "n_sessions", "n_negative",
       "minutes_score"]

GIU = set()
for goc in CAU_HINH:
    for seed in (0, 1, 2):
        GIU.add("%s_seed%d" % (goc, seed))

dong, da_thay = [], set()
for s in glob.glob(tam + "/**/summary.csv", recursive=True):
    for r in csv.DictReader(open(s)):
        rid = r.get("run_id", "")
        goc = re.sub(r"_(tong|val_[A-Z]{2})$", "", rid)
        # PHẢI lọc cả experiment: tn1_lstm.zip nén cả runs/tn1 lẫn runs/tn1_ghij,
        # mà config_id của hai bên giống hệt nhau. Chỉ lọc theo config_id thì ba
        # dòng test GHIJ lọt vào bảng TN1.
        if r.get("experiment") != "tn1" or goc not in GIU or rid in da_thay:
            continue
        da_thay.add(rid)
        r["device"] = ""
        dong.append({k: r.get(k, "") for k in COT})

# Tệp nén không phải lúc nào cũng có đủ dòng TONG: run_cv.py ghi dòng đó sau
# khi xong cả bốn fold, nên nếu ô lưu kết quả chạy trước thì dòng TONG của seed
# cuối chưa kịp vào. Dựng lại từ scores.csv per-fold — cùng đúng công thức
# run_cv.py dùng: macro = trung bình bốn fold, std = độ lệch chuẩn TỔNG THỂ.
co = {r["run_id"] for r in dong}
for goc, ten in CAU_HINH.items():
    for seed in (0, 1, 2):
        rid = "%s_seed%d_tong" % (goc, seed)
        thu_muc = "%s/%s/seed%d" % (DICH, ten, seed)
        if rid in co or not os.path.isdir(thu_muc):
            continue
        v = [macro_cua_fold("%s/%s/scores.csv" % (thu_muc, f))[0] for f in FOLDS]
        mau = next((r for r in dong
                    if r["run_id"] == "%s_seed%d_val_AB" % (goc, seed)), None)
        if mau is None:
            continue
        moi = {k: "" for k in COT}
        moi.update({k: mau[k] for k in ("git_commit", "experiment", "model",
                                        "loss", "alpha", "corr_threshold",
                                        "n_params", "epochs")})
        moi.update({"run_id": rid, "seed": str(seed), "fold": "TONG",
                    "val_users": "ABCDEFKL", "n_sessions": "4",
                    "score_macro": repr(statistics.mean(v)),
                    "score_std": repr(statistics.pstdev(v))})
        dong.append(moi)
        print("dựng lại dòng TONG thiếu:", rid, "-> %.6f" % statistics.mean(v))

dong.sort(key=lambda r: (r["model"], int(r["n_params"]), int(r["seed"]), r["fold"]))
os.makedirs("/content/runs_TN1/tn1", exist_ok=True)
with open("/content/runs_TN1/tn1/summary_bo_sung.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COT)
    w.writeheader(); w.writerows(dong)

from collections import Counter
print("gom", len(dong), "dòng")
for k, v in sorted(Counter((r["model"], r["n_params"]) for r in dong).items()):
    print("   %-10s %9s tham số   %2d dòng" % (k[0], k[1], v))
print()
print("còn tên GPU:", sum(1 for r in dong if r["device"]))

## 8. Đóng gói và tải về

In [ ]:
subprocess.run(["bash", "-lc",
                "cd /content && tar -czf runs_TN1.tar.gz runs_TN1"], check=True)
print("%.1f MB" % (os.path.getsize("/content/runs_TN1.tar.gz") / 1e6))
print()
print(subprocess.run(["bash", "-lc",
      "cd /content && find runs_TN1 -maxdepth 3 | sort | head -30"],
      capture_output=True, text=True).stdout)

In [ ]:
from google.colab import files
files.download("/content/runs_TN1.tar.gz")

## 9. Ở máy

```bash
tar -xzf runs_TN1.tar.gz
cp -r runs_TN1/tn1/LSTM-352 runs_TN1/tn1/LSTM-67 runs_TN1/tn1/CNN-LSTM-58 runs/tn1/
python3 scripts/gop_summary.py runs_TN1/tn1/summary_bo_sung.csv
python3 scripts/compare_cv.py --experiment tn1
```

`gop_summary.py` không đè dòng đã có trong repo — dòng trùng `run_id` thì giữ
bản cũ và báo ra, để một lần chạy lại không lặng lẽ thay số đã công bố.

Ô 6 đã tính lại điểm từ `scores.csv` và so với bảng, nên nếu ô đó in
`TẤT CẢ KHỚP BẢNG` thì tệp tải về dùng được luôn.